In [ ]:
# Phase 1 — CNN Backbone Benchmark
# NOTE: Training is SKIPPED here (set epochs=0).
# To train, set epochs=50 and run on Colab GPU.
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/lung_nodule_processed/labels.csv')
print(f"Dataset: {len(df)} nodules")


In [ ]:
# Architecture check: verify all backbones instantiate and forward-pass correctly
from src.models.backbones import BackboneClassifier
import torch

BACKBONES = ['mobilenet_v3_large', 'efficientnet_b0', 'resnet50', 'densenet121', 'convnext_tiny']
x = torch.randn(2, 3, 64, 64)  # (batch, 3-slice, H, W)

for name in BACKBONES:
    model = BackboneClassifier(name, n_input_channels=3, pretrained=False)
    out = model(x)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name}: output={out.shape}, params={n_params/1e6:.1f}M")


In [ ]:
# TRAINING CELL — set SKIP_TRAINING=False to actually train on Colab GPU
SKIP_TRAINING = True  # set False to train

if not SKIP_TRAINING:
    from torch.utils.data import DataLoader
    from src.training.dataset import NoduleDataset2_5D
    from src.training.trainer import run_kfold_cv
    from src.models.backbones import BackboneClassifier
    from src.evaluation.metrics import aggregate_fold_results
    import yaml, os

    cfg = yaml.safe_load(open('/content/lung-nodule-fusion-xai/configs/train.yaml'))
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    results = {}
    for backbone_name in BACKBONES:
        def model_factory():
            return BackboneClassifier(backbone_name, n_input_channels=3, pretrained=True)
        def dataset_factory(sub_df, augment):
            ds = NoduleDataset2_5D(sub_df, augment=augment)
            return DataLoader(ds, batch_size=16, shuffle=augment, num_workers=2, pin_memory=True)

        fold_results = run_kfold_cv(
            model_factory=model_factory,
            dataset_factory=dataset_factory,
            labels_df=df,
            epochs=cfg['training']['epochs'],
            device_str=device,
            checkpoint_dir=f'/content/drive/MyDrive/checkpoints/{backbone_name}',
        )
        results[backbone_name] = fold_results

    print("Training complete")
else:
    print("Training skipped (SKIP_TRAINING=True). Set False to train on GPU.")
